In [ ]:
'''
figure out how many transient candidates get cut at each stage
'''

In [1]:
import numpy as np
import pandas as pd
from astropy.io import fits
import matplotlib.pyplot as plt
import precision_recall as pr
import gc  # garbage collector
import review_findings_functions as rff
import os
import importlib
import implantation_functions as imp
# import corner
plt.rcParams["figure.figsize"] = (10, 7)
plt.rcParams["image.interpolation"] = "nearest"
plt.rcParams["image.cmap"] = "gist_gray"

In [7]:
len(files)

14353

In [6]:
#let's count the number of pixels >.99
folder_path = "/lustre/lrspec/ZTF_data/science_extractions/1020april26/average/"
pixels_count = 0
transients_count = 0
# # List all the files in the folder
files = [f for f in os.listdir(folder_path) if f.endswith('alllist') and not(f.endswith('29._0_alllist') or f.endswith('29._1_alllist') or f.endswith('29._3_alllist') or f.endswith('29._4_alllist'))] 

# for file in files:
#     it = pd.read_csv(folder_path+file)
#     pixels_count+= np.sum(it['detection_size'])
#     transients_count+=len(it)
#     del it
#     gc.collect()
    

# print("pixels, transients ", pixels_count, transients_count )                        

In [8]:
for file in files:
    it = pd.read_csv(folder_path+file)
    pixels_count+= np.sum(it['detection_size'])
    transients_count+=len(it)
    del it
    gc.collect()
    

print("pixels, transients ", pixels_count, transients_count )                        

pixels, transients  5197200 868249


In [14]:
print("pixels, transients ", pixels_count, transients_count )   

pixels, transients  1114378 186499


In [16]:
df

,Unnamed: 0.1,Unnamed: 0,filename,quad,type,row_from_nn,col_from_nn,detection_size,closest_x,closest_y,Diff
0,0,0,ztf_20221020141806_000686_zi_c02_x_0012.,2,pos,63.500000,306.000000,8,316.0,2099.0,0.000000
1,1,1,ztf_20221020141806_000686_zi_c02_x_0012.,2,pos,490.375000,1895.375000,8,1904.0,2531.0,1.375000
2,2,2,ztf_20221020141806_000686_zi_c02_x_0012.,2,pos,935.000000,787.000000,1,797.0,938.0,0.000000
3,3,3,ztf_20221020141806_000686_zi_c02_x_0012.,2,pos,1595.000000,817.000000,3,824.0,1449.0,3.000000
4,4,4,ztf_20221020141806_000686_zi_c02_x_0012.,2,pos,1670.000000,650.000000,1,662.0,1501.0,2.000000
5,5,5,ztf_20221020141806_000686_zi_c02_x_0012.,2,pos,1721.000000,534.000000,1,541.0,1825.0,3.000000
6,6,6,ztf_20221020141806_000686_zi_c02_x_0012.,2,pos,1912.000000,359.000000,1,368.0,1048.0,1.000000
7,7,7,ztf_20221020141806_000686_zi_c02_x_0012.,2,pos,1912.500000,1182.000000,2,1193.0,1749.0,1.000000
8,8,8,ztf_20221020141806_000686_zi_c02_x_0012.,2,pos,1912.500000,1184.000000,2,1193.0,1749.0,1.000000
9,9,9,ztf_20221020141806_000686_zi_c02_x_0012.,2,pos,1924.000000,280.000000,15,290.0,1821.0,0.000000


In [11]:
#now let's count the diffcut but only the good ones
#we can do size 1 here as well
predictions = pd.read_csv('/lustre/lrspec/ZTF_data/random_forest_training/predictions_updated.csv', names=['filename', 'rf'])

folder_path = "/lustre/lrspec/ZTF_data/science_extractions/1020april26/average/"
transients_count_diffcut = 0
transients_count_diffcut_nosize1 = 0
# # List all the files in the folder
files = [f for f in os.listdir(folder_path) if f.endswith('diffcut5')and not(f.endswith('29._0_alllist_matched_diffcut5') or f.endswith('29._1_alllist_matched_diffcut5') or f.endswith('29._3_alllist_matched_diffcut5') or f.endswith('29._4_alllist_matched_diffcut5'))] 

In [10]:
files[:10]

['ztf_20221020141806_000686_zi_c02_x_0012._2_alllist_matched_diffcut5',
 'ztf_20221020163819_000640_zg_c14_x_0010._3_alllist_matched_diffcut5',
 'ztf_20221020141806_000686_zi_c13_x_0011._1_alllist_matched_diffcut5',
 'ztf_20221020148032_000686_zg_c08_x_0008._1_alllist_matched_diffcut5',
 'ztf_20221020148032_000686_zg_c08_x_0012._4_alllist_matched_diffcut5',
 'ztf_20221020157697_000640_zi_c05_x_0009._4_alllist_matched_diffcut5',
 'ztf_20221020157697_000640_zi_c05_x_0013._1_alllist_matched_diffcut5',
 'ztf_20221020294271_000449_zr_c10_x_0005._3_alllist_matched_diffcut5',
 'ztf_20221020157697_000640_zi_c14_x_0010._2_alllist_matched_diffcut5',
 'ztf_20221020294271_000449_zr_c04_x_0007._4_alllist_matched_diffcut5']

In [13]:


for file in files:
    #figure out whether it's in the random forest
    df = pd.read_csv(folder_path+file)
    for filename, group in df.groupby('filename'):     
        for quadnum, quadgroup in group.groupby('quad'):
            for starx, stargroup in quadgroup.groupby('closest_x'):
                lc_filename = 'lightcurve_'+filename[:-1]+'_'+str(quadnum)+'_'+f'{int(starx):04}'+'_'+f'{int(stargroup.iloc[0]["closest_y"]):04}'+'.csv'
                # lc_filename = 'lightcurve_'+df_row['filename'][:-1]+'_'+str(df_row['quadrant'])+'_'+f'{int(df_row["closest_x"]):04}'+'_'+f'{int(df_row["closest_y"]):04}'+'.csv'
                matching_row = predictions[predictions['filename']==lc_filename]
                # print(lc_filename)
                # print(matching_row['rf'].values[0])
                if len(matching_row)==1:
                    if matching_row['rf'].values[0]==0:
                        transients_count_diffcut+=len(stargroup)
                        transients_count_diffcut_nosize1+=len(stargroup[stargroup['detection_size']>1])
                elif len(matching_row)==0:
                    # concat.loc[i, 'random_forest_result']=None #actually can just leave this as a NaN
                    
                    print('didnt find it')
                    print(matching_row)
                else:
                    
                    print('something strange is happening...')
                    print(matching_row)
                    print(len(matching_row))
    
    del df
    gc.collect()
    

print("transients_count_diffcut, transients_count_diffcut_nosize1 ", transients_count_diffcut, transients_count_diffcut_nosize1)    

didnt find it
Empty DataFrame
Columns: [filename, rf]
Index: []
transients_count_diffcut, transients_count_diffcut_nosize1  1147156 808772


In [ ]:
#573578 404386

In [14]:
868249-573578

294671

In [15]:
573578-404386

169192

In [25]:
print("transients_count_diffcut, transients_count_diffcut_nosize1 ", transients_count_diffcut, transients_count_diffcut_nosize1)    

transients_count_diffcut, transients_count_diffcut_nosize1  577 403


In [ ]:
#finally count deduplicated
predictions = pd.read_csv('/lustre/lrspec/ZTF_data/random_forest_training/predictions_updated.csv', names=['filename', 'rf'])

folder_path = "/lustre/lrspec/ZTF_data/science_extractions/1020april26/average/"
transients_count_dedup = 0
# transients_count_diffcut_nosize1 = 0
# # List all the files in the folder
files = [f for f in os.listdir(folder_path) if f.endswith('dedup') and not(f.endswith('29._0_alllist_matched_diffcut5_size1_dedup') or f.endswith('29._1_alllist_matched_diffcut5_size1_dedup') or f.endswith('29._3_alllist_matched_diffcut5_size1_dedup') or f.endswith('29._4_alllist_matched_diffcut5_size1_dedup'))]  


In [26]:

for file in files:
    #figure out whether it's in the random forest
    df = pd.read_csv(folder_path+file)
    for filename, group in df.groupby('filename'):     
        for quadnum, quadgroup in group.groupby('quad'):
            for starx, stargroup in quadgroup.groupby('closest_x'):
                lc_filename = 'lightcurve_'+filename[:-1]+'_'+str(quadnum)+'_'+f'{int(starx):04}'+'_'+f'{int(stargroup.iloc[0]["closest_y"]):04}'+'.csv'
                # lc_filename = 'lightcurve_'+df_row['filename'][:-1]+'_'+str(df_row['quadrant'])+'_'+f'{int(df_row["closest_x"]):04}'+'_'+f'{int(df_row["closest_y"]):04}'+'.csv'
                matching_row = predictions[predictions['filename']==lc_filename]
                # print(lc_filename)
                # print(matching_row['rf'].values[0])
                if len(matching_row)==1:
                    if matching_row['rf'].values[0]==0:
                        transients_count_dedup+=len(stargroup)
                elif len(matching_row)==0:
                    # concat.loc[i, 'random_forest_result']=None #actually can just leave this as a NaN
                    
                    print('didnt find it')
                    print(matching_row)
                else:
                    
                    print('something strange is happening...')
                    print(matching_row)
                    print(len(matching_row))
    
    del df
    gc.collect()
    

print("transients_count_dedup", transients_count_dedup)    

KeyboardInterrupt: 

In [27]:
print("transients_count_dedup", transients_count_dedup)    

transients_count_dedup 112


In [17]:
404386 - 328464

75922

In [29]:
np.save('/home/3155/counts.npy', np.array([pixels_count, transients_count, transients_count_diffcut, transients_count_diffcut_nosize1, transients_count_dedup]))

In [16]:
np.load('/home/3155/counts.npy')

array([573578, 404386, 328464])